In [21]:
# ============================================================
#  JOUR 10 / 30 DAYS OF PYTHON — Dashboard Plotly Interactif
#  Concepts : plotly.express · go.Figure · make_subplots
#             hover · couleurs · export HTML + PNG (matplotlib)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import plotly.express       as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

os.makedirs("outputs", exist_ok=True)

# ─────────────────────────────────────────────────────────────
#  1. DONNÉES
# ─────────────────────────────────────────────────────────────

try:
    df = pd.read_csv('ventes_propres.csv', parse_dates=['date'])
    print(f"✅ ventes_propres.csv chargé : {df.shape[0]} lignes")
except FileNotFoundError:
    np.random.seed(42)
    n = 50
    df = pd.DataFrame({
        'date':          pd.date_range('2024-01-01', periods=n, freq='3D'),
        'produit':       np.random.choice(['Laptop Pro','Smartphone X',
                                           'Tablette Air','Écouteurs BT',
                                           'Montre Smart'], n),
        'region':        np.random.choice(['Île-de-France','PACA',
                                           'Auvergne-Rhône-Alpes',
                                           'Grand Est'], n),
        'quantite':      np.random.randint(1, 12, n).astype(float),
        'prix_unitaire': np.random.choice([1200, 650, 450, 120, 280], n),
        'montant':       np.zeros(n),
    })
    df['montant'] = df['quantite'] * df['prix_unitaire']
    print(f"✅ Dataset généré : {df.shape[0]} lignes")

df['mois'] = df['date'].dt.to_period('M').astype(str)

ca_produit = df.groupby('produit')['montant'].sum().reset_index()
ca_region  = df.groupby('region')['montant'].sum().reset_index()
ca_mensuel = df.groupby('mois')['montant'].sum().reset_index()

COLORS = ['#00FF94','#00E0FF','#A78BFA','#FFD700','#FF6B6B']
BG     = '#0F172A'
CARD   = '#1E293B'
TEXT   = '#E2E8F0'
GRID   = '#334155'

# ─────────────────────────────────────────────────────────────
#  HELPERS matplotlib
# ─────────────────────────────────────────────────────────────

def dark_fig(w=10, h=6, title=''):
    fig, ax = plt.subplots(figsize=(w, h))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TEXT)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    ax.set_title(title, color=TEXT, fontsize=14, pad=12)
    return fig, ax

def save_png(fig, name):
    path = f'outputs/{name}'
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'✅ PNG → {path}')

# ─────────────────────────────────────────────────────────────
#  2. GRAPHIQUE 1 — Bar chart (matplotlib PNG) + Plotly HTML
# ─────────────────────────────────────────────────────────────

data1 = ca_produit.sort_values('montant', ascending=True)
fig, ax = dark_fig(10, 5, 'CA par Produit')
bars = ax.barh(data1['produit'], data1['montant'],
               color=COLORS[:len(data1)])
ax.set_xlabel('Montant (€)', color=TEXT)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
ax.grid(axis='x', color=GRID, linestyle='--', alpha=0.5)
for bar, val in zip(bars, data1['montant']):
    ax.text(val + max(data1['montant']) * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}€', va='center', color=TEXT, fontsize=9)
save_png(fig, 'g1_bar_produit.png')

fig1 = px.bar(data1, x='montant', y='produit', orientation='h',
              title='CA par Produit — hover pour les détails',
              color='produit', color_discrete_sequence=COLORS, text='montant')
fig1.update_traces(texttemplate='%{text:,.0f}€', textposition='outside',
                   hovertemplate='<b>%{y}</b><br>CA : %{x:,.0f}€<extra></extra>')
fig1.update_layout(paper_bgcolor=BG, plot_bgcolor=CARD, font=dict(color=TEXT),
                   showlegend=False, xaxis=dict(gridcolor=GRID, tickformat=',.0f'),
                   yaxis=dict(gridcolor=GRID), title_font_size=16)
fig1.write_html('outputs/g1_bar_interactif.html')
print('✅ HTML → outputs/g1_bar_interactif.html')

# ─────────────────────────────────────────────────────────────
#  3. GRAPHIQUE 2 — Donut chart (matplotlib PNG) + Plotly HTML
# ─────────────────────────────────────────────────────────────

fig, ax = dark_fig(8, 6, 'Répartition CA par Région')
wedges, texts, autotexts = ax.pie(
    ca_region['montant'], labels=ca_region['region'],
    colors=COLORS[:len(ca_region)], autopct='%1.1f%%',
    startangle=90, wedgeprops=dict(width=0.55),
    textprops=dict(color=TEXT))
for at in autotexts:
    at.set_color(BG)
    at.set_fontweight('bold')
save_png(fig, 'g2_pie_region.png')

fig2 = px.pie(ca_region, values='montant', names='region',
              title='Répartition CA par Région',
              color_discrete_sequence=COLORS, hole=0.4)
fig2.update_traces(hovertemplate='<b>%{label}</b><br>CA : %{value:,.0f}€<br>Part : %{percent}<extra></extra>',
                   textinfo='label+percent')
fig2.update_layout(paper_bgcolor=BG, font=dict(color=TEXT), title_font_size=16)
fig2.write_html('outputs/g2_pie_interactif.html')
print('✅ HTML → outputs/g2_pie_interactif.html')

# ─────────────────────────────────────────────────────────────
#  4. GRAPHIQUE 3 — Line chart (matplotlib PNG) + Plotly HTML
# ─────────────────────────────────────────────────────────────

fig, ax = dark_fig(12, 5, 'Évolution Mensuelle du CA')
ax.plot(ca_mensuel['mois'], ca_mensuel['montant'],
        color=COLORS[0], linewidth=2.5, marker='o', markersize=7)
ax.fill_between(ca_mensuel['mois'], ca_mensuel['montant'],
                alpha=0.15, color=COLORS[0])
ax.set_xlabel('Mois', color=TEXT)
ax.set_ylabel('Montant (€)', color=TEXT)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
ax.grid(color=GRID, linestyle='--', alpha=0.5)
plt.xticks(rotation=35, ha='right', color=TEXT)
save_png(fig, 'g3_line_mensuel.png')

fig3 = px.line(ca_mensuel, x='mois', y='montant',
               title='Évolution Mensuelle — glisse le range slider',
               markers=True, color_discrete_sequence=[COLORS[0]])
fig3.update_traces(hovertemplate='<b>%{x}</b><br>CA : %{y:,.0f}€<extra></extra>',
                   line=dict(width=3), marker=dict(size=9))
fig3.update_xaxes(rangeslider_visible=True, rangeslider_thickness=0.08)
fig3.update_layout(paper_bgcolor=BG, plot_bgcolor=CARD, font=dict(color=TEXT),
                   xaxis=dict(gridcolor=GRID),
                   yaxis=dict(gridcolor=GRID, tickformat=',.0f', ticksuffix='€'),
                   title_font_size=16)
fig3.write_html('outputs/g3_line_slider.html')
print('✅ HTML → outputs/g3_line_slider.html')

# ─────────────────────────────────────────────────────────────
#  5. GRAPHIQUE 4 — Scatter plot (matplotlib PNG) + Plotly HTML
# ─────────────────────────────────────────────────────────────

fig, ax = dark_fig(10, 6, 'Quantité vs Montant — taille = valeur de la vente')
produits = df['produit'].unique()
for i, prod in enumerate(produits):
    sub = df[df['produit'] == prod]
    sizes = (sub['montant'] / sub['montant'].max() * 300).clip(lower=30)
    ax.scatter(sub['quantite'], sub['montant'],
               color=COLORS[i % len(COLORS)], s=sizes,
               alpha=0.75, label=prod, edgecolors='none')
ax.set_xlabel('Quantité vendue', color=TEXT)
ax.set_ylabel('Montant (€)', color=TEXT)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
ax.grid(color=GRID, linestyle='--', alpha=0.5)
legend = ax.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT, fontsize=9)
save_png(fig, 'g4_scatter_quantite.png')

fig4 = px.scatter(df, x='quantite', y='montant', color='produit', size='montant',
                  hover_name='produit',
                  hover_data={'region': True, 'date': '|%d/%m/%Y', 'montant': ':,.0f'},
                  title='Quantité vs Montant — taille = valeur de la vente',
                  color_discrete_sequence=COLORS)
fig4.update_layout(paper_bgcolor=BG, plot_bgcolor=CARD, font=dict(color=TEXT),
                   xaxis=dict(gridcolor=GRID, title='Quantité vendue'),
                   yaxis=dict(gridcolor=GRID, tickformat=',.0f',
                              ticksuffix='€', title='Montant (€)'),
                   title_font_size=16, legend=dict(bgcolor=CARD, bordercolor=GRID))
fig4.write_html('outputs/g4_scatter_interactif.html')
print('✅ HTML → outputs/g4_scatter_interactif.html')

# ─────────────────────────────────────────────────────────────
#  6. DASHBOARD PNG — grille 2×2 matplotlib
# ─────────────────────────────────────────────────────────────

fig_dash = plt.figure(figsize=(18, 10))
fig_dash.patch.set_facecolor(BG)
fig_dash.suptitle('Dashboard Ventes — Plotly / matplotlib',
                  color=TEXT, fontsize=18, fontweight='bold', y=1.01)
gs = GridSpec(2, 2, figure=fig_dash, hspace=0.4, wspace=0.35)

# ── Panel 1 : Bar
ax1 = fig_dash.add_subplot(gs[0, 0])
ax1.set_facecolor(CARD)
d = ca_produit.sort_values('montant', ascending=True)
bars = ax1.barh(d['produit'], d['montant'], color=COLORS[:len(d)])
ax1.set_title('CA par Produit', color=TEXT, fontsize=12)
ax1.tick_params(colors=TEXT, labelsize=8)
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k€'))
ax1.grid(axis='x', color=GRID, linestyle='--', alpha=0.5)
for sp in ax1.spines.values(): sp.set_edgecolor(GRID)

# ── Panel 2 : Donut
ax2 = fig_dash.add_subplot(gs[0, 1])
ax2.set_facecolor(CARD)
wedges, _, autotexts = ax2.pie(
    ca_region['montant'], labels=ca_region['region'],
    colors=COLORS[:len(ca_region)], autopct='%1.0f%%',
    startangle=90, wedgeprops=dict(width=0.55),
    textprops=dict(color=TEXT, fontsize=8))
for at in autotexts:
    at.set_color(BG); at.set_fontweight('bold'); at.set_fontsize(8)
ax2.set_title('CA par Région', color=TEXT, fontsize=12)

# ── Panel 3 : Line
ax3 = fig_dash.add_subplot(gs[1, 0])
ax3.set_facecolor(CARD)
ax3.plot(ca_mensuel['mois'], ca_mensuel['montant'],
         color=COLORS[0], linewidth=2, marker='o', markersize=5)
ax3.fill_between(ca_mensuel['mois'], ca_mensuel['montant'],
                 alpha=0.15, color=COLORS[0])
ax3.set_title('Évolution Mensuelle', color=TEXT, fontsize=12)
ax3.tick_params(colors=TEXT, labelsize=7)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k€'))
ax3.grid(color=GRID, linestyle='--', alpha=0.5)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=35, ha='right')
for sp in ax3.spines.values(): sp.set_edgecolor(GRID)

# ── Panel 4 : Scatter
ax4 = fig_dash.add_subplot(gs[1, 1])
ax4.set_facecolor(CARD)
for i, prod in enumerate(df['produit'].unique()):
    sub = df[df['produit'] == prod]
    sizes = (sub['montant'] / df['montant'].max() * 200).clip(lower=20)
    ax4.scatter(sub['quantite'], sub['montant'],
                color=COLORS[i % len(COLORS)], s=sizes, alpha=0.75,
                label=prod, edgecolors='none')
ax4.set_title('Quantité vs Montant', color=TEXT, fontsize=12)
ax4.tick_params(colors=TEXT, labelsize=8)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k€'))
ax4.grid(color=GRID, linestyle='--', alpha=0.5)
legend = ax4.legend(facecolor=CARD, edgecolor=GRID, labelcolor=TEXT, fontsize=7)
for sp in ax4.spines.values(): sp.set_edgecolor(GRID)

fig_dash.savefig('outputs/dashboard_plotly.png', dpi=150,
                 bbox_inches='tight', facecolor=BG)
plt.close(fig_dash)
print('✅ Dashboard PNG → outputs/dashboard_plotly.png')

# ─────────────────────────────────────────────────────────────
#  7. DASHBOARD HTML interactif (Plotly)
# ─────────────────────────────────────────────────────────────

dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=['CA par Produit','CA par Région',
                    'Évolution Mensuelle','Quantité vs Montant'],
    specs=[[{'type':'bar'},{'type':'pie'}],
           [{'type':'scatter'},{'type':'scatter'}]],
)

for i, row in ca_produit.sort_values('montant').iterrows():
    dashboard.add_trace(
        go.Bar(x=[row['montant']], y=[row['produit']], orientation='h',
               marker_color=COLORS[i % len(COLORS)], showlegend=False,
               hovertemplate=f"<b>{row['produit']}</b><br>{row['montant']:,.0f}€<extra></extra>"),
        row=1, col=1)

dashboard.add_trace(
    go.Pie(labels=ca_region['region'], values=ca_region['montant'],
           marker_colors=COLORS[:len(ca_region)], hole=0.4, showlegend=False,
           hovertemplate='<b>%{label}</b><br>%{value:,.0f}€<extra></extra>'),
    row=1, col=2)

dashboard.add_trace(
    go.Scatter(x=ca_mensuel['mois'], y=ca_mensuel['montant'],
               mode='lines+markers', showlegend=False,
               line=dict(color=COLORS[0], width=3), marker=dict(size=8),
               hovertemplate='<b>%{x}</b><br>%{y:,.0f}€<extra></extra>'),
    row=2, col=1)

for i, prod in enumerate(df['produit'].unique()):
    sub = df[df['produit'] == prod]
    dashboard.add_trace(
        go.Scatter(x=sub['quantite'], y=sub['montant'], mode='markers',
                   name=prod, showlegend=False,
                   marker=dict(color=COLORS[i % len(COLORS)], size=10, opacity=0.75),
                   hovertemplate=f'<b>{prod}</b><br>Qté : %{{x}}<br>%{{y:,.0f}}€<extra></extra>'),
        row=2, col=2)

dashboard.update_layout(
    title=dict(text='Dashboard Ventes Interactif — Plotly',
               font=dict(size=20, color=TEXT)),
    paper_bgcolor=BG, plot_bgcolor=CARD,
    font=dict(color=TEXT, size=11), height=800, width=1400, showlegend=False)
dashboard.update_xaxes(gridcolor=GRID, color='#94A3B8')
dashboard.update_yaxes(gridcolor=GRID, color='#94A3B8')
dashboard.write_html('outputs/dashboard_plotly.html')
print('✅ Dashboard HTML → outputs/dashboard_plotly.html')

print('\n🎉 Tous les fichiers sont dans outputs/')

✅ ventes_propres.csv chargé : 49 lignes
✅ PNG → outputs/g1_bar_produit.png
✅ HTML → outputs/g1_bar_interactif.html
✅ PNG → outputs/g2_pie_region.png
✅ HTML → outputs/g2_pie_interactif.html
✅ PNG → outputs/g3_line_mensuel.png
✅ HTML → outputs/g3_line_slider.html
✅ PNG → outputs/g4_scatter_quantite.png
✅ HTML → outputs/g4_scatter_interactif.html
✅ Dashboard PNG → outputs/dashboard_plotly.png
✅ Dashboard HTML → outputs/dashboard_plotly.html

🎉 Tous les fichiers sont dans outputs/
